# Tutorial 3_1: Dynamic Fully Coupled Thermoelasticity



In this tutorial, we will consider a slender cantilever beam undergoing dynamic fully coupled thermoelasticity. Unlike the previous quasi-static heat conduction example, this problem accounts for inertial effects (acceleration) and thermal-mechanical coupling solved monolithically. We will observe the beam "snap back" from an initially bent state and watch the thermoelastic damping effect.

# Standard imports

In [ ]:
try:
    from firedrake import *
except ImportError:
    !wget "https://fem-on-colab.github.io/releases/firedrake-install-release-real.sh" -O "/tmp/firedrake-install.sh" && bash "/tmp/firedrake-install.sh"
    from firedrake import *

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import ArtistAnimation
from IPython.display import HTML

from tqdm.auto import tqdm

# Mesh and Function Spaces

We define a slender cantilever beam (10 units long, 1 unit high). Because the physics are fully coupled, we use a mixed monolithic function space `W` containing both the displacement vector space `V_u` and the temperature space `V_T`.

In [ ]:
Lx = 10.0
Ly = 1.0
nx = 200
ny = 20

mesh = RectangleMesh(nx, ny, Lx, Ly, quadrilateral=True)

# Function spaces: CG2 for displacement and temperature
V_T = FunctionSpace(mesh, "CG", 1)
V_u = VectorFunctionSpace(mesh, "CG", 1)
W = V_u * V_T  # Mixed monolithic space

V_dg = FunctionSpace(mesh, "DG", 0)

output_deformation_factor = 1.0
coords = mesh.coordinates.dat.data.copy()

# Material Properties and Initial Conditions

To simulate the dynamic "snap-back", we initialize the beam with a parabolic downward deflection but zero initial velocity. We achieve this by applying the initial deflection to both the current state ($n-1$) and the previous state ($n-2$).

In [ ]:
dt = Constant(0.05)
n_steps = 200
T_ref = Constant(300)

# Material properties
rho = Constant(1.0)       # Material density for inertia
rho_c = Constant(0.1)     # Volumetric heat capacity
k = Constant(0.05)        # Thermal conductivity
E = Constant(1e5)         # Young's Modulus
nu = Constant(0.3)        # Poisson's ratio
alpha = Constant(5e-4)    # Thermal expansion coefficient

lmbda = (E * nu) / ((1 + nu) * (1 - 2 * nu))
mu = E / (2 * (1 + nu))
beta = alpha * (3 * lmbda + 2 * mu)

# Mixed state functions for time-stepping
w = Function(W, name="State_n")        # Current step (n)
w_1 = Function(W, name="State_n-1")    # Previous step (n-1)
w_2 = Function(W, name="State_n-2")    # Older step (n-2)

u, T = split(w)
u_1, T_1 = split(w_1)
u_2, T_2 = split(w_2)

# Initialize temperature field to reference temperature
w.sub(1).interpolate(T_ref)
w_1.sub(1).interpolate(T_ref)
w_2.sub(1).interpolate(T_ref)

# Apply an initial "bent" displacement
x, y = SpatialCoordinate(mesh)
initial_deflection = as_vector([0.0, -2 * (x/Lx)**2])
w_1.sub(0).project(initial_deflection)
w_2.sub(0).project(initial_deflection) # u_1 = u_2 means initial velocity is zero

# Boundary Conditions: Fixed and isothermal left edge
bc_u = DirichletBC(W.sub(0), Constant((0.0, 0.0)), 1)
bc_T = DirichletBC(W.sub(1), T_ref, 1)
bcs = [bc_u, bc_T]

# Monolithic Solver Setup

## Mathematical Derivation

### 1. Strong Form
The dynamic mechanical equilibrium and coupled heat equations are:
$$\rho \frac{\partial^2 u}{\partial t^2} - \nabla \cdot \sigma = 0$$
$$\rho_c \frac{\partial T}{\partial t} - \nabla \cdot (k \nabla T) + \beta T_{ref} \frac{\partial (\nabla \cdot u)}{\partial t} = 0$$

### 2. Time Discretisation
We discretise the time derivatives using finite difference schemes:
Acceleration: $$\frac{\partial^2 u}{\partial t^2} \approx \frac{u^{n} - 2u^{n-1} + u^{n-2}}{\Delta t^2}$$
Rates (Velocity / Temperature): $$\frac{\partial (\cdot)}{\partial t} \approx \frac{(\cdot)^{n} - (\cdot)^{n-1}}{\Delta t}$$

### 3. Weak Form
Multiplying by test functions $v_u$ and $v_T$ and integrating by parts yields the combined monolithic weak form:
$$\int_{\Omega} \left( \rho \frac{u^{n} - 2u^{n-1} + u^{n-2}}{\Delta t^2} \cdot v_u + \sigma(u^n, T^n) : \epsilon(v_u) \right) dx + \int_{\Omega} \left( \rho_c \frac{T^n - T^{n-1}}{\Delta t} v_T + k \nabla T^n \cdot \nabla v_T + \beta T_{ref} \frac{\nabla \cdot u^n - \nabla \cdot u^{n-1}}{\Delta t} v_T \right) dx = 0$$

In [ ]:
def epsilon(v):
    return sym(grad(v))

def sigma_expr(v, temp):
    dim = mesh.topological_dimension()
    return lmbda * tr(epsilon(v)) * Identity(dim) + 2 * mu * epsilon(v) - beta * (temp - T_ref) * Identity(dim)

v_u, v_T = TestFunctions(W)

# Finite difference time derivatives
accel_u = (u - 2*u_1 + u_2) / (dt**2)
div_u_rate = (div(u) - div(u_1)) / dt
T_rate = (T - T_1) / dt

# Dynamic Mechanical Equilibrium
F_u = (inner(rho * accel_u, v_u) + inner(sigma_expr(u, T), epsilon(v_u))) * dx

# Coupled Heat Equation
F_T = (rho_c * T_rate * v_T + inner(k * grad(T), grad(v_T)) + beta * T_ref * div_u_rate * v_T) * dx

# Global residual
F = F_u + F_T

prob = NonlinearVariationalProblem(F, w, bcs=bcs)
solver = NonlinearVariationalSolver(prob)

# Animation setup

We setup an animation to visualize both the transient temperature and the von Mises stress fields over time.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6))

for ax, title in zip([ax1, ax2], ["Temperature Evolution", "Deformed von Mises Stress"]):
    ax.set_aspect('equal')
    ax.set_title(title)
    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")

frames = []

# Stress calculation helper
vm_stress = Function(V_dg, name="von_Mises_Stress")
def get_von_mises(v, temp):
    s = sigma_expr(v, temp)
    return sqrt(s[0,0]**2 + s[1,1]**2 - s[0,0]*s[1,1] + 3*s[0,1]**2)

plt.tight_layout()
plt.close()

# Solver Loop

In [ ]:
# Space to project CG2 displacement onto CG1 for mesh coordinate deformation mapping
V_cg1 = VectorFunctionSpace(mesh, "CG", 1)
u_cg1 = Function(V_cg1)

for step in tqdm(range(n_steps), desc="Solving Dynamic System"):
    solver.solve()
    
    u_out, T_out = w.subfunctions
    vm_stress.project(get_von_mises(u_out, T_out))
    
    # Project CG2 displacement to CG1 to safely deform the CG1 mesh coordinates for plotting
    u_cg1.project(u_out)
    mesh.coordinates.dat.data[:] = coords + u_cg1.dat.data[:] * output_deformation_factor
    
    c1 = tripcolor(T_out, axes=ax1, cmap='inferno')
    c2 = tripcolor(vm_stress, axes=ax2, cmap='coolwarm')
    frames.append([c1, c2])
    
    # Reset mesh coordinates back to original
    mesh.coordinates.dat.data[:] = coords
    
    # Shift states for next time step
    w_2.assign(w_1)
    w_1.assign(w)

# Generate the animation

In [ ]:
plt.close()
ani = ArtistAnimation(fig, frames, interval=50, blit=True, repeat_delay=1000)
HTML(f'<div style="width:100%;">{ani.to_jshtml()}</div>')